# Ejercicio de clase: Predicción del precio de vivienda con Decision Tree Regressor

**Dataset**: California Housing Prices — https://www.kaggle.com/datasets/camnugent/california-housing-prices

En el ejemplo de clase (`decision_tree_example.ipynb`) usamos un **Decision Tree Classifier** para predecir
si un paciente sobrevivía o no a una sepsis. Esa era una tarea de **clasificación**: la variable que
queríamos predecir (`hospital_outcome`) solo podía tomar un número limitado de valores (0 o 1, "vive" o
"muere").

En este ejercicio vamos a resolver un problema distinto: **regresión**. Vamos a predecir el **precio
mediano de una vivienda** (`median_house_value`) en un bloque censal de California, a partir de
características como la ubicación, la cantidad de habitaciones o el ingreso medio de sus habitantes.

> **Diferencia clave**: en clasificación el modelo predice una *categoría* (una clase). En regresión el
> modelo predice un *número real* (puede tomar, en principio, cualquier valor dentro de un rango continuo).
> Esto tiene consecuencias importantes: no podemos usar métricas como *precision*, *recall* o *F1*, porque
> esas métricas comparan clases exactas. En su lugar usaremos métricas que midan **qué tan lejos** está la
> predicción del valor real, como el **MAE (Mean Absolute Error)**, que veremos más adelante.

Para este ejercicio usarán `DecisionTreeRegressor` en lugar de `DecisionTreeClassifier`.

### Antes de empezar

1. Descarguen el dataset desde Kaggle: https://www.kaggle.com/datasets/camnugent/california-housing-prices
2. El archivo se llama `housing.csv`. Colóquenlo dentro de la carpeta `raw/` de este proyecto
   (la misma carpeta `decision_tree/raw/` donde está el dataset de sepsis).
3. Sigan cada sección en orden. Cada sección tiene:
   - Una breve explicación de qué vamos a hacer y por qué.
   - Un **reto**: ustedes deben escribir el código (no está resuelto).
   - **Preguntas de análisis**: respóndanlas en una celda de markdown justo debajo de su código.


### Importar librerías

Igual que en el ejemplo de clase, empezamos importando las librerías que vamos a necesitar. Noten dos
diferencias frente al ejemplo de clasificación:

- Usamos `DecisionTreeRegressor` en lugar de `DecisionTreeClassifier`.
- Usamos `mean_absolute_error` en lugar de `precision_score`, `recall_score`, `f1_score`.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error


## Paso 1 — Cargar los datos

**Reto**: Carguen el archivo `housing.csv` (carpeta `raw/`) en un DataFrame de pandas llamado `df`, tal
como hicimos en el ejemplo de clase con `pd.read_csv(...)`. Luego muestren las primeras filas del
DataFrame para confirmar que se cargó correctamente.


In [3]:
# TODO: cargar el dataset en un DataFrame llamado df
df = pd.read_csv('raw/housing.csv')

In [4]:
# TODO: mostrar las primeras filas del DataFrame
print(df.head(10))
print()

# Miremos cuántas filas y columnas tiene el dataSet
print("Tamaño del DT:", df.shape)

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   
5    -122.25     37.85                52.0        919.0           213.0   
6    -122.25     37.84                52.0       2535.0           489.0   
7    -122.25     37.84                52.0       3104.0           687.0   
8    -122.26     37.84                42.0       2555.0           665.0   
9    -122.25     37.84                52.0       3549.0           707.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0   

**Preguntas de análisis**

**1. ¿Cuántas filas y cuántas columnas tiene el dataset? (pista: `df.shape`)**

El Dataset tiene 20 640 filas (samples) y 10 columnas (features)

**2. ¿Qué representa cada fila del dataset? ¿Es una vivienda individual o algo distinto?**

Es una casa específica en California con las características de sí misma y de sus habitantes.

**3. Observando los nombres de las columnas, ¿cuál creen que es la variable que vamos a predecir (el *target*)?**

Debido a que el objetivo es predecir el precio promedio, teniendo en cuenta el dataSet, *Target* es:

   $$
   median house value
   $$

Cuya posición corresponde con el de la 9na columna

## Paso 2 — Identificación inicial de los datos

Antes de tocar cualquier dato, siempre debemos entender con qué estamos trabajando: cuántas columnas hay,
qué tipo de dato tiene cada una, si hay valores nulos, y cuál es el rango de valores de cada variable.

**Reto**: Usando lo que ya conocen de pandas, respondan (con código) estas tres preguntas:
1. ¿Qué tipo de dato (`dtype`) tiene cada columna? ¿Hay alguna columna categórica (texto)?
2. ¿Cuáles son las estadísticas básicas (media, desviación estándar, mínimo, máximo, etc.) de las
   columnas numéricas?
3. ¿Hay valores nulos (faltantes) en el dataset? ¿En qué columna(s)?

Pistas de los métodos que necesitan (ya los usamos, en otra forma, en el ejemplo de clase): `.info()`,
`.describe()`, `.isnull()`.


In [5]:
# TODO: tipos de dato de cada columna
print(df.dtypes)

# La ultima columna, contiene datos tipo objeto que en la practica se traducen como strings finalmente

longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity        object
dtype: object


In [6]:
# TODO: estadísticas descriptivas de las columnas numéricas
print(df.describe())

          longitude      latitude  housing_median_age   total_rooms  \
count  20640.000000  20640.000000        20640.000000  20640.000000   
mean    -119.569704     35.631861           28.639486   2635.763081   
std        2.003532      2.135952           12.585558   2181.615252   
min     -124.350000     32.540000            1.000000      2.000000   
25%     -121.800000     33.930000           18.000000   1447.750000   
50%     -118.490000     34.260000           29.000000   2127.000000   
75%     -118.010000     37.710000           37.000000   3148.000000   
max     -114.310000     41.950000           52.000000  39320.000000   

       total_bedrooms    population    households  median_income  \
count    20433.000000  20640.000000  20640.000000   20640.000000   
mean       537.870553   1425.476744    499.539680       3.870671   
std        421.385070   1132.462122    382.329753       1.899822   
min          1.000000      3.000000      1.000000       0.499900   
25%        296.00000

In [7]:
# TODO: cantidad de valores nulos por columna
amt = pd.isnull(df).sum()
print("Cantidad de valores nulos por columna:")
print(amt)

# Como se observa, la única columna con valores nulos es "Total_bedrooms" que tiene 207

Cantidad de valores nulos por columna:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


**Preguntas de análisis**

**1. ¿Cuál es la única columna categórica (de texto) del dataset? ¿Qué valores puede tomar?**

Es *ocean_proximity*, y puede tomar los valores: [NEAR BAY, <1H OCEAN, INLAND]

**2. ¿Qué columna tiene valores nulos? ¿Cuántas filas están afectadas, aproximadamente qué porcentaje del total representa?**

La única columna con valores nulos es total_bedrooms, con un total de 207 filas afectada (representa alrededor de un 1% del total)

**3. Miren el `min` y el `max` de `median_house_value`. ¿Les parece un rango razonable para el precio de una vivienda? ¿Notan algo raro en el valor máximo? (pista: busquen cuántas filas tienen exactamente ese valor máximo).**


In [8]:
#Miremos cuantas filas de la columna median_house_value tienen el valor máximo (500 001 dolares)
cnt = pd.Series(df['median_house_value'] == 500001).sum()
print(cnt)

965


Los valores difieren en general mucho, y el valor máximo solo lo tiene alrededor del 4% de las casas por lo que indica que los datos tienen una desviación estándar bastante alta

4. Comparen el rango de `median_income` con el de `total_rooms`. ¿Están en escalas muy distintas? ¿Creen
   que eso sería un problema para un Decision Tree? (piensen en cómo el árbol elige los cortes: ¿necesita
   que las variables estén en la misma escala, como sí lo necesitan otros modelos?)

Teniendo en cuenta que 

$$
Rango = MaxValue - MinValue
$$

Rango de `median_income`: 15.000100 - 0.499900: 14,5011

Rango de `total_rooms`: 39320.000000 - 2.000000 : 39 319

A pesar de que los rangos de los datos tienen escalas que difieren considerablemente, esto no representa una dificultad para la clasificación del modelo pues las desiciones se van tomando sin relacionar los datos entre sí. considerando cada uno individualmente.

## Paso 3 — Procesamiento de datos

Por ahora, para mantener el ejercicio simple, vamos a trabajar **solo con variables numéricas**. Más
adelante en el curso aprenderemos técnicas para incorporar variables categóricas (como *one-hot encoding*),
pero hoy las vamos a descartar.

**Reto**:
1. Eliminen del DataFrame la(s) columna(s) categórica(s) que identificaron en el paso anterior.
2. Decidan qué hacer con los valores nulos que encontraron (por ejemplo, eliminar esas filas) y
   apliquen esa decisión. Un Decision Tree Regressor de scikit-learn no puede entrenarse si quedan
   valores `NaN` en los datos.
3. Confirmen, con código, que ya no quedan columnas categóricas ni valores nulos.


In [9]:
# TODO: eliminar la(s) columna(s) categórica(s)
df_num = df.drop('ocean_proximity', axis=1)

In [10]:
# TODO: manejar los valores nulos (por ejemplo, eliminarlos)
df_num = df_num.dropna(subset=["total_bedrooms"])

In [11]:
# TODO: confirmar que no quedan nulos ni columnas categóricas
amt = pd.isnull(df_num).sum()
print("Cantidad de valores nulos por columna:")
print(amt)
print()

print("Columnas restantes:")

print(df_num.head(5))

print()
print(df_num.shape)

Cantidad de valores nulos por columna:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64

Columnas restantes:
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value  
0       322.0       126.0         8.3252            452600.0  
1      2401.0      1138.0         8.3014            358500.0  
2       496.0       177.0         7.2574            352100.0  
3       

**Preguntas de análisis**
1. ¿Qué información de las viviendas estamos perdiendo al eliminar la columna categórica? ¿Creen que esa
   información podría ser útil para predecir el precio? ¿Por qué?

Estamos perdiendo la distancia al mar de la casa, es un valor importante pues en la realidad, la cercanía al mar normalmente represente un precio mayor.

2. Si eliminaron filas con nulos, ¿cuántas filas quedaron en total? ¿Qué porcentaje del dataset original
   se perdió?

Al eliminar las filas con valores nulos, quedaron en total 20433 filas. Se perdió alrededor del 1% de la información del dataFrame

3. ¿Qué otra estrategia (distinta a eliminar las filas) existe para tratar valores nulos? ¿Por qué hoy
   optamos por la más simple?

Nomrlamente se reemplaza ese valor nulo con un dato estadístico, ya sea la media, la moda, la mediana, el dato vecino, etc.
Este procedimiento, el foco no está específicamente en esto, por lo que simplemente se eliminan


## Paso 4 — Definir variables predictoras (X) y variable objetivo (y), y dividir los datos

Igual que en el ejemplo de clase, necesitamos separar:
- **X**: las columnas que el modelo usará para predecir (todas menos el precio).
- **y**: la columna que queremos predecir (`median_house_value`).

Y luego dividir ambas en un conjunto de **entrenamiento** y uno de **prueba**, usando
`train_test_split`, tal como hicimos con los datos de sepsis.

**Reto**:
1. Construyan `X` (todas las columnas numéricas excepto `median_house_value`) y `y`
   (`median_house_value`).
2. Usen `train_test_split` para crear `X_train`, `X_test`, `y_train`, `y_test`. Usen un `test_size` de
   0.2 y `random_state=0` para que los resultados sean reproducibles.


In [12]:
# TODO: construir X (predictoras) y y (target)

# En X van a quedar todas las columnas menos la columna target (median_house_value)
columnasX = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
X = df_num[columnasX].copy()
print(X.head(5))
print(X.shape)
print()

# En Y solo queda el target
y = df_num.drop(columns=columnasX)
print(y.head(5))
print(y.shape)
print()


   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  
0       322.0       126.0         8.3252  
1      2401.0      1138.0         8.3014  
2       496.0       177.0         7.2574  
3       558.0       219.0         5.6431  
4       565.0       259.0         3.8462  
(20433, 8)

   median_house_value
0            452600.0
1            358500.0
2            352100.0
3            341300.0
4            342200.0
(20433, 1)



In [13]:
# TODO: dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(X_train.shape)
print(X_test.shape)

(16346, 8)
(4087, 8)


**Preguntas de análisis**
**1. ¿Cuántas filas quedaron en `X_train` y cuántas en `X_test`?**

En X_train quedaron 16 346 y en X_test 4 087. (Cumpliendo una razón de división 80% - 20%)

**2. ¿Por qué es importante evaluar el modelo en datos que **no** usó para entrenar (`X_test`, `y_test`)? ¿Qué pasaría si evaluáramos únicamente sobre `X_train`?**

Porque de esta manera se considera el acoplamiento que puede desarollar el modelo a los datos de entrenamiento. Esto se traduce en que el modelo se vuelve muy preciso para clasificar a los datos de entrenamiento, pero para datos los datos de prueba (que no tuvo en cuenta para entrenar) puede diferir en el resultado.

**3. En el ejemplo de clase balanceamos las clases con SMOTE antes de dividir los datos. Aquí no lo hicimos. ¿Por qué SMOTE (que genera ejemplos sintéticos de una *clase* minoritaria) no tiene sentido en un problema de regresión, donde no hay clases sino un valor continuo?**

Porque al ser valores continuos (infinitas clases), realmente SMOTE no tendría sentido ni funcionaría como en un problema de clasificación. (Es muy complicado balancear n clases cuando n es un número muy grande) 

## Paso 5 — Entrenar el Decision Tree Regressor

**Reto**: Entrenen un `DecisionTreeRegressor` (con `random_state=0`) usando `X_train` y `y_train`, de la
misma forma en que entrenaron el `DecisionTreeClassifier` en el ejemplo de clase.


In [14]:
# TODO: crear y entrenar el modelo
reg = DecisionTreeRegressor(random_state=0)
reg.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


**Preguntas de análisis**

**1. En clasificación, cada hoja del árbol predice una clase (por ejemplo, "vive" o "muere"). En regresión, ¿qué creen que predice cada hoja del árbol? (pista: piensen en los valores de `y` que caen en esa hoja durante el entrenamiento).**

Las hojas en regresión probablemente representen diferentes intervalos de números reales, es probablemente de esta manera teninedo en cuenta que son números reales y hacer una hoja por cada uno, es simplemente imposible

**2. ¿Qué criterio usa por defecto `DecisionTreeRegressor` para decidir dónde hacer cada corte, en lugar del *gini* o *entropy* que se usan en clasificación? (revisen la documentación del parámetro `criterion`).**

Por defecto, DecisionTreeRegressor usa el criterio squared_error.
Este criterio busca realizar los cortes que minimicen el error cuadrático medio (MSE) dentro de los nodos resultantes. Es decir, el árbol intenta que los valores reales de cada grupo queden lo más cerca posible de su media.

## Paso 6 — Evaluar el modelo con MAE

### ¿Qué es el MAE (Mean Absolute Error)?

El **MAE** es el promedio de la diferencia absoluta entre el valor real y el valor predicho:

$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$

A diferencia de precision/recall/F1 (que solo tienen sentido cuando comparamos clases), el MAE funciona
sobre valores numéricos continuos y nos dice, **en promedio, cuánto se equivoca el modelo, en las mismas
unidades que la variable objetivo**. En nuestro caso, como `median_house_value` está en dólares, un
MAE de, por ejemplo, 40000 significa que en promedio el modelo se equivoca en $40,000 dólares al predecir
el precio de una vivienda.

Un MAE más bajo es mejor. Pero un mismo valor de MAE puede ser "bueno" o "malo" dependiendo de la escala
de la variable que estamos prediciendo — por eso siempre hay que compararlo contra algo (por ejemplo,
el precio promedio de las viviendas).

**Reto**:
1. Usen el modelo entrenado para predecir sobre `X_train` y calculen el MAE comparando esas predicciones
   con `y_train`.
2. Hagan lo mismo sobre `X_test` con `y_test`.
3. Comparen ambos valores.


In [15]:
# TODO: predicciones y MAE sobre el set de entrenamiento

# Usemos el arbol que ya entrenamos, para predecir el valor de Y con los datos de entrenamiento (X_train)
y_train_pred = reg.predict(X_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
print("MAE (train):", mae_train)


MAE (train): 0.0


In [16]:
# TODO: predicciones y MAE sobre el set de prueba
y_test_pred = reg.predict(X_test)
mae_test = mean_absolute_error(y_test, y_test_pred)
print("MAE (test):", mae_test)


MAE (test): 43557.103743577194


**Preguntas de análisis**

**1. ¿El MAE de entrenamiento es mayor, menor o similar al MAE de prueba? ¿Qué les dice eso sobre qué tan bien "memorizó" el árbol los datos de entrenamiento?**

Es muchisimo menor, es muy interesante de echo mirar que para los datos de entrenamiento el MAE es 0. Indicando que el modelo se adaptó demasiado a los datos de entrenamiento (los memorizó). Por el contrario, cuando se revisa el MAE para los datos de prueba se evidencia que es muchisimo mayor mostrando que para datos diferentes el modelo es mas impreciso. 

**2. Calculen el precio promedio (`.mean()`) de `median_house_value` en todo el dataset. Comparando ese promedio con el MAE de prueba, ¿el error del modelo les parece grande o pequeño en proporción al precio típico de una vivienda?**

In [18]:
#Saquemos el promedio de los valores de la columna de median_house_value

promedio = df['median_house_value'].mean()
print(promedio)


206855.81690891474


Realmente el MAE para los datos de prueba considerando el precio promedio sí es bastante alto, pues un error de 44 000 dolares aprox. sí representa un gran desnivel de clasificación.


**3. En el ejemplo de clase, un árbol sin restricciones (`fully grown`) mostraba señales de sobreajuste (*overfitting*) al compararlo con un árbol más simple (`max_depth=3`). Según los MAE de train y test que obtuvieron, ¿creen que este árbol también está sobreajustado? ¿Por qué?**

Sí, el árbol presenta señales de sobreajuste. El MAE de entrenamiento es 0.0, lo que significa que el árbol logra predecir perfectamente los datos de train, mientras que en test el MAE aumenta a aproximadamente 43,557. Esta gran diferencia indica que el árbol se ajustó demasiado a los datos de entrenamiento y tiene una menor capacidad de generalización a datos nuevos.

## Reto adicional (opcional) — ¿Se puede mejorar el árbol?

En el ejemplo de clase, limitar la profundidad del árbol (`max_depth=3`) cambió el comportamiento del
modelo. Prueben lo mismo aquí:

1. Entrenen un segundo `DecisionTreeRegressor`, esta vez fijando `max_depth` (prueben con distintos
   valores, por ejemplo 3, 5, 10).
2. Calculen el MAE de train y de test para cada valor de `max_depth`.
3. Grafiquen (opcional) el MAE de train y de test contra `max_depth`, como una curva.

**Preguntas de análisis**
1. ¿Qué le pasa al MAE de entrenamiento a medida que aumentan `max_depth`? ¿Y al de prueba?
2. ¿Existe un valor de `max_depth` donde el MAE de prueba deja de mejorar (o empeora)? ¿Qué relación tiene
   eso con el concepto de *overfitting* que vimos en el ejemplo de clase?
3. Miren `reg.feature_importances_` del árbol entrenado. ¿Cuáles son las 2 o 3 variables más importantes
   para predecir el precio de la vivienda? ¿Tiene sentido con lo que ustedes esperarían intuitivamente?


In [ ]:
# Espacio libre para el reto adicional
